In [ ]:
import pandas as pd

df = pd.read_csv("/content/Dataset proyecto .csv")

cols_texto = df.select_dtypes(include='object').columns

#Nulos

print("-"*45)
print(f" Nulos")
print("-"*45)

print("="*45)
nulos = df.isnull().sum().sum()
print(f" Cantidad nulos           : {nulos}")
print("="*45)

print(" Columnas con nulos")
print("="*45)
nulos = df.isnull().sum()
print(nulos[nulos > 0])
print("="*45)



# Mal registrados (celdas con textos de error)

print("-"*45)
print(f" Mal registrados")
print("-"*45)

print("="*45)
mal_registrados = 0
for col in cols_texto:
    mal_registrados += df[col].str.lower().str.startswith('error').sum()
print(f"  Mal registrados          : {mal_registrados}")
print("="*45)

print("-"*45)
print(f" Inconsistencias")
print("-"*45)
# inconsistencias
columnas = cols_texto.tolist()

for col in columnas:
    valores = df[col].dropna().unique()
    encontrados = {}

    for val in valores:
        key = val.strip().lower()
        if key not in encontrados:
            encontrados[key] = []
        encontrados[key].append(val)

    total = sum(1 for g in encontrados.values() if len(g) > 1)

    print(f"{'='*45}")
    print(f"  {col} {" "*((5+18)-(len(col)))}:{total}")
    print(f"{'='*45}")

    for key, grupo in encontrados.items():
        if len(grupo) > 1:
            print(f"  '{key}' aparece como: {grupo}")

    if total == 0:
        print("  Sin inconsistencias")

print(" "*45)
df.tail(1)

---------------------------------------------
 Nulos
---------------------------------------------
 Cantidad nulos           : 6
 Columnas con nulos
CLASIFICACIÓN TAXONOMÍA    2
TIPO DE INCIDENTE          2
IMPORTANCIA                2
dtype: int64
---------------------------------------------
 Mal registrados
---------------------------------------------
  Mal registrados          : 3
---------------------------------------------
 Inconsistencias
---------------------------------------------
  FECHA DE SOLICITUD      :0
  Sin inconsistencias
  ORDEN DE LA ENTIDAD     :0
  Sin inconsistencias
  SECTOR                  :2
  'ambiente y desarrollo sostenible' aparece como: ['Ambiente y Desarrollo Sostenible', 'Ambiente Y Desarrollo Sostenible']
  'minas y energía' aparece como: ['Minas y Energía', 'Minas Y Energía']
  REGIÓN                  :0
  Sin inconsistencias
  DEPARTAMENTO            :0
  Sin inconsistencias
  CLASIFICACIÓN TAXONOMÍA :0
  Sin inconsistencias
  TIPO DE INCIDENTE  

,FECHA DE SOLICITUD,ORDEN DE LA ENTIDAD,SECTOR,REGIÓN,DIVIPOLA DEPARTAMENTO,DEPARTAMENTO,CLASIFICACIÓN TAXONOMÍA,TIPO DE INCIDENTE,IMPORTANCIA
5226,2026 Apr 01 12:00:00 AM,Nacional,Hacienda y Crédito Público,Bogotá D.C.,11,Bogotá D.C.,Fraude,Phishing,Menor


Limpiando nulos

In [ ]:
df1 = df.copy()
notnu = df1[((df1['CLASIFICACIÓN TAXONOMÍA'].notnull())
        &(df1['TIPO DE INCIDENTE'].notnull())
        &(df1['IMPORTANCIA'].notnull()))].copy()

print(notnu.isnull().sum())

FECHA DE SOLICITUD         0
ORDEN DE LA ENTIDAD        0
SECTOR                     0
REGIÓN                     0
DIVIPOLA DEPARTAMENTO      0
DEPARTAMENTO               0
CLASIFICACIÓN TAXONOMÍA    0
TIPO DE INCIDENTE          0
IMPORTANCIA                0
dtype: int64


arreglando inconsitencias

In [ ]:
sectores = df1["SECTOR"].unique()
incidentes = df1['TIPO DE INCIDENTE'].unique()
print("inconsistencias en sectores = ",len(sectores))
print("inconsistencias en incidentes = ", len(incidentes))

df2 = notnu
#---------------------------------------------------------------
df2["SECTOR"] = df2['SECTOR'].str.lower()
df2["TIPO DE INCIDENTE"] = df2['TIPO DE INCIDENTE'].str.lower()
#---------------------------------------------------------------
sec = []
inc = []
for i in sectores:
  dato = i.lower()
  if dato not in sec:
    sec.append(dato)

for j in sectores:
  dato2 = j.lower()
  if dato2 not in inc:
    inc.append(dato)
print("sectores =", len(sec))
print("Tipo incidente = ",len(inc))

inconsistencias en sectores =  30
inconsistencias en incidentes =  27
sectores = 28
Tipo incidente =  29


Eliminar datos con error

In [ ]:
df3 = df2.copy()
mask = df3.map(lambda x: isinstance(x, str) and x.lower().startswith('error')).any(axis=1)
print("Filas con error:", mask.sum())
print("Filas antes:", len(df3))
df3 = df3[~mask]
print("Filas después:", len(df3))

Filas con error: 3
Filas antes: 5225
Filas después: 5222


In [ ]:
dfnew = df3.copy()

#crear nuevas columnas

dfnew["DIA"] = dfnew['FECHA DE SOLICITUD'].str.slice(9,11)
dfnew['MES'] = dfnew['FECHA DE SOLICITUD'].str.slice(5,8)
dfnew["AÑO"] = dfnew["FECHA DE SOLICITUD"].str.slice(0,4)

#eliminar datos del 2026

dfnew = dfnew[dfnew['AÑO'] != "2026"]
print("lista años =",dfnew["AÑO"].unique())

#eliminar columna de fecha
dfnew = dfnew.drop(columns=["FECHA DE SOLICITUD"])

dfnew.tail(5)


lista años = ['2019' '2020' '2021' '2022' '2023' '2024' '2025']


,ORDEN DE LA ENTIDAD,SECTOR,REGIÓN,DIVIPOLA DEPARTAMENTO,DEPARTAMENTO,CLASIFICACIÓN TAXONOMÍA,TIPO DE INCIDENTE,IMPORTANCIA,DIA,MES,AÑO
5111,Privada,privado,Bogotá D.C.,11,Bogotá D.C.,Fraude,phishing,Menor,22,Dec,2025
5112,Nacional,función pública,Bogotá D.C.,11,Bogotá D.C.,Fraude,uso no autorizado de recursos,Menor,22,Dec,2025
5113,Privada,privado,Bogotá D.C.,11,Bogotá D.C.,Fraude,phishing,Menor,23,Dec,2025
5114,Nacional,educación,Bogotá D.C.,11,Bogotá D.C.,Fraude,phishing,Menor,26,Dec,2025
5115,Privada,privado,Bogotá D.C.,11,Bogotá D.C.,Fraude,phishing,Grave,31,Dec,2025


In [ ]:
# Save the dfnew DataFrame to a new CSV file
dfnew.to_csv('datasetclean.csv', index=False, sep=';', encoding='utf-8-sig')
print("DataFrame 'dfnew' successfully saved to 'dfnew_cleaned.csv'")

DataFrame 'dfnew' successfully saved to 'dfnew_cleaned.csv'
